# mutation_burden.ipynb
Do ecDNA+ tumors have greater somatic mutation burden?  
Do ecDNA+ tumors have greater pathogenic somatic mutation burden?  
We conpared variant count and stratified ecDNA by tumor type.

RESULTS  
The somatic variant count of ecDNA+ tumors was significantly higher than that of ecDNA- tumors.  
The pathogenic somatic variant count of ecDNA+ tumors was significantly higher than that of ecDNA- tumors.  
HGG and MBL has enough samples to compare.In both tumor types, significant difference was observed in somatic variant count. However, significant difference was not observed in pathogenic somatic variant count.

## Import and format 
our data into a DataFrame indexed by biosample and with columns  
\[ 	cancer_type 	amplicon_class 	all_counts 	pathogenic_counts \]

In [ ]:
import pandas as pd

In [ ]:
def parse_bcftools_stats(file):
    # read a bcftools stats output, and
    # return the total number of variants
    ## YOUR CODE HERE
    with open(file) as f:
        for line in f:
            if line.startswith('SN	0	number of records:'):
                count=line.rstrip().split('\t')[-1]
                count=int(count)
    return count

In [ ]:
import os
def get_variant_counts(path):
    # for all .stats.txt files in path,
    # create a dictionary of the format:
    # return {sample : count}
    d = {}
    for filename in os.listdir(path):
        full_path = os.path.join(path, filename)
        if full_path.endswith('.stats.txt'):
            count = parse_bcftools_stats(full_path)
            #print(count)
            filename = filename.split('.')[0]
            d[filename] = count
    return d

def get_variant_count_df():
    # return a dataframe of the format:
    # file_id, all_counts, pathogenic_counts

    # get all counts
    all_counts = get_variant_counts('./data/stats/all')
    df1 = pd.Series(all_counts)
    df1.name = "all_counts"
    df = pd.DataFrame(df1)
    # get pathogenic counts
    pathogenic_counts = get_variant_counts('./data/stats/pathogenic')
    df2 = pd.Series(pathogenic_counts)
    df2.name = "pathogenic_counts"
    # check the indices are the same
    assert set(df1.index) == set(df2.index)
    # merge and return
    df = df.join(df2)
    return df

In [ ]:
get_variant_count_df().head()

In [ ]:
def read_manifest(file='./data/pedpancan/manifest_20250910_143948.csv'):
    df = pd.read_csv(file)
    df=df[df.name.str.endswith('.gz')]
    df['index'] = df.name.map(lambda x: os.path.basename(x).split('.')[0])
    df = df.set_index('index')
    df = df[['Kids First Biospecimen ID']].copy()
    return df
read_manifest()

In [ ]:
def read_biosample_table(file='./data/pedpancan/Supplementary Tables.xlsx'):
    df = pd.read_excel(file,sheet_name='2. Biosamples',index_col='biosample_id')
    df = df[df.in_unique_patient_set].copy()
    return df

def merge_counts(manifest,counts):
    assert set(manifest.index) == set(counts.index)
    df = manifest.join(counts)
    return df

def check_file_bs_ids(example):
    # check no duplicates
    assert len(example.index) == len(example.index.unique())
    assert len(example['Kids First Biospecimen ID']) == len(example['Kids First Biospecimen ID'].unique())
    # check no missing values
    assert sum(example.index.isna()) == 0
    assert sum(example['Kids First Biospecimen ID'].isna()) == 0
    # check 1:1 mapping
    assert len(example.index) == len(example['Kids First Biospecimen ID'])
    return

def merge_ecDNA_counts(ecDNA_df,counts_df):
    check_file_bs_ids(counts_df)
    counts_df['file_id'] = counts_df.index
    counts_df = counts_df.set_index('Kids First Biospecimen ID')
    df = ecDNA_df.join(counts_df,how='inner')
    return df

In [ ]:
df = merge_ecDNA_counts(read_biosample_table(),
                   merge_counts(read_manifest(),get_variant_count_df())
                  )
df.head()

In [ ]:
df = df.loc[:,['cancer_type','amplicon_class','all_counts','pathogenic_counts']]
df

## Tests and plots
Compare mutation burden across amplicon classes by Wilcoxon rank sum test,
make boxplots

In [ ]:
import scipy
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def compare_somatic_mutation_burden_osc(df, test=scipy.stats.mannwhitneyu):
    df['log_counts'] = np.log1p(df.all_counts)
    sns.boxplot(
        data = df,
        x = 'amplicon_class',
        #y = 'all_counts'
        y = 'log_counts'
    )

    classes = list(df.amplicon_class.unique()) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['log_counts']
            b = df[df.amplicon_class == classes[j]]['log_counts']
            print(test(a,b))
    return

In [ ]:
compare_somatic_mutation_burden_osc(df)
plt.ylabel('Log(Number of somatic mutations)')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/compare_somatic_mutation_burden.svg')
plt.show()

In [ ]:
def compare_pathogenic_somatic_mutation_burden(df, test=scipy.stats.mannwhitneyu):
    
    sns.violinplot(
        data = df,
        x = 'amplicon_class',
        y = 'pathogenic_counts',
        density_norm = 'area'
    )
    sns.despine()
    classes = list(df.amplicon_class.unique()) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['pathogenic_counts']
            b = df[df.amplicon_class == classes[j]]['pathogenic_counts']
            print(test(a,b))
    return

In [ ]:
compare_pathogenic_somatic_mutation_burden(df)
plt.ylabel('Number of pathogenic mutations in each patient')
plt.savefig('figure/compare_pathogenic_somatic_mutation_burden.svg')
plt.show()

In [ ]:
def stratify_ecDNA_by_tumor_type(df):
    df = df.loc[:,['amplicon_class','cancer_type']]
    df = df.groupby(['cancer_type','amplicon_class']).size()
    return df

In [ ]:
stratify_ecDNA_by_tumor_type(df)

In [ ]:
df_MBL = df[df['cancer_type'].isin(['MBL'])].copy()
#df_MBL = df_MBL[df_MBL['amplicon_class'].isin(['no amplification','ecDNA'])]
df_MBL

In [ ]:
df_HGG = df[df['cancer_type'].isin(['HGG'])].copy()
df_HGG

In [ ]:
compare_somatic_mutation_burden_osc(df_MBL)
plt.ylabel('Log(Number of somatic mutations in each MBL patient)')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/MBL_compare_somatic_mutation_burden.svg')
plt.show()

In [ ]:
compare_pathogenic_somatic_mutation_burden(df_MBL)
plt.ylabel('Number of pathogenic mutations in each MBL patient')
plt.savefig('figure/MBL_compare_pathogenic_somatic_mutation_burden.svg')
plt.show()

In [ ]:
compare_somatic_mutation_burden_osc(df_HGG)
plt.ylabel('Log(Number of somatic mutations in each HGG patient)')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/HGG_somatic_mutation_burden.svg')
plt.show()

In [ ]:
compare_pathogenic_somatic_mutation_burden(df_HGG)
plt.ylabel('Number of pathogenic mutations in each HGG patient')
plt.savefig('figure/HGG_pathogenic_somatic_mutation_burden.svg')
plt.show()